# Explorer les distributions

Comparer des moyennes ne décrit pas l’étalement ni les valeurs atypiques. Le boxplot complète les indicateurs, sans remplacer l’analyse du grain.

**Question métier :** Deux catégories aux profits moyens proches ont-elles les mêmes profits par ligne ?

Les données sont montées en lecture seule. Enregistrez vos figures dans `/workspace/resultats`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '/workspace/data/sample_-_superstore.xls'
orders = pd.read_excel(DATA, sheet_name='Orders', engine='xlrd')
orders['Order Date'] = pd.to_datetime(orders['Order Date'])

## Notion 1 — Lire les quartiles d’un boxplot


### Étape 1 — Préparer une observation par ligne

Ici le grain reste la ligne de commande ; les profits sont comparés par catégorie.

In [ ]:
categories = sorted(orders['Category'].unique())
series = [orders.loc[orders['Category'].eq(c),'Profit'].dropna() for c in categories]
print([s.median() for s in series])

### Étape 2 — Construire le boxplot

La boîte décrit les quartiles ; les points éloignés restent des observations à vérifier.

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.boxplot(series, tick_labels=categories, orientation='horizontal', showfliers=True)
ax.axvline(0, color='black', linewidth=1)
ax.set(title='Profit par ligne de commande', xlabel='Profit (unités monétaires)')
plt.show()

### Étape 3 — Interpréter

La médiane de Technology est plus élevée, mais le boxplot montre aussi l’étalement et les valeurs atypiques. Chaque observation reste une ligne de commande.

### Exercice 1 — Décrire une catégorie

Choisissez une catégorie et relevez sa médiane, ses quartiles et le nombre de lignes à profit négatif.

**À rendre :** Quatre valeurs ou constats et une phrase qui nomme le grain.

In [ ]:
s = orders.loc[orders['Category'].eq('Technology'),'Profit']
print('Médiane :',s.median())
print('Quartiles :',s.quantile([.25,.75]))
print('Lignes négatives :',(s<0).sum())

### Mise en commun

Les indicateurs décrivent les profits par ligne de commande. Ils ne donnent pas directement le profit par commande complète.

## Notion 2 — Changer le grain : une observation par commande


### Étape 1 — Additionner les lignes d’une commande

Order ID regroupe toutes les lignes d’une commande avant de tracer sa distribution.

In [ ]:
profit_commande = orders.groupby('Order ID')['Profit'].sum()
print(profit_commande.describe())

### Étape 2 — Comparer les distributions

Le boxplot porte maintenant sur les commandes agrégées.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.boxplot([profit_commande], tick_labels=['Commandes'], orientation='horizontal', showfliers=True)
ax.axvline(0, color='black', linewidth=1)
ax.set(title='Profit par commande', xlabel='Profit (unités monétaires)')
plt.show()

### Étape 3 — Interpréter

Le grain change : chaque observation résume désormais un Order ID. Une commande peut réunir des lignes positives et négatives.

### Exercice 2 — Comparer les profits négatifs

Combien de commandes ont un profit total négatif ? Quelle part des lignes a un profit négatif ?

**À rendre :** Deux comptes et deux proportions calculés au bon grain.

In [ ]:
profit_commande = orders.groupby('Order ID')['Profit'].sum()
print('Commandes négatives :',(profit_commande<0).sum())
print('Part des commandes :',(profit_commande<0).mean()*100)
print('Part des lignes :',(orders['Profit']<0).mean()*100)

### Mise en commun

Environ 1 040 commandes sur 5 111 ont un profit négatif (20,3 %). La part des lignes négatives est différente, car le grain change.

## Notion 3 — Comparer médiane et moyenne


### Étape 1 — Calculer les deux repères

La médiane résiste davantage aux valeurs extrêmes ; la moyenne les intègre.

In [ ]:
resume = orders.groupby('Category')['Profit'].agg(mediane='median', moyenne='mean')
print(resume)

### Étape 2 — Représenter les résultats

Une comparaison côte à côte fait apparaître les écarts entre les deux indicateurs.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
y = range(len(resume))
ax.barh([i-.18 for i in y], resume['mediane'], height=.36, label='Médiane')
ax.barh([i+.18 for i in y], resume['moyenne'], height=.36, label='Moyenne')
ax.set_yticks(list(y), resume.index)
ax.set(title='Profit moyen et médian par ligne', xlabel='Unités monétaires')
ax.legend()
plt.show()

### Étape 3 — Interpréter

Un écart important entre moyenne et médiane peut signaler une distribution asymétrique ou influencée par des valeurs extrêmes ; il ne prouve pas une erreur.

### Exercice 3 — Calculer un écart moyen-médian

Dans quelle catégorie la moyenne est-elle la plus éloignée de la médiane ?

**À rendre :** La catégorie, les deux valeurs et une interprétation sans attribuer de cause.

In [ ]:
resume = orders.groupby('Category')['Profit'].agg(mediane='median', moyenne='mean')
resume['ecart'] = (resume['moyenne']-resume['mediane']).abs()
print(resume.loc[resume['ecart'].idxmax()])

### Mise en commun

La moyenne est sensible aux valeurs extrêmes et peut s’éloigner de la médiane. Les deux statistiques décrivent les profits des lignes de commande.